## Hücre 1 — Kütüphaneleri Kur

In [ ]:
!pip install qdrant-client sentence-transformers -q
print('✓ Kurulum tamamlandı')

## Hücre 2 — API Key ve Bağlantı Bilgileri
Sol menü → 🔑 Secrets → `QDRANT_API_KEY` eklenmiş olmalı.

In [ ]:
from google.colab import userdata

QDRANT_URL     = "https://9fc39cdf-6767-47d6-99a6-cc2a7c711424.us-west-1-0.aws.cloud.qdrant.io"
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")  # Secrets'tan güvenli okuma
KOLEKSIYON_ADI = "kira_hukuku"

print('✓ Bilgiler yüklendi')
print(f'  URL    : {QDRANT_URL}')
print(f'  API Key: {QDRANT_API_KEY[:12]}...')  # güvenlik için kısalt

## Hücre 3 — JSON'ı Yükle

In [ ]:
%pip install PyMuPDF

In [ ]:
import fitz
import json
import re

# PDF adını tam olarak yüklediğin dosyaya göre ayarladık
pdf_yolu = "./tbk borclar kanunu.pdf"
json_yolu = "tbk borclar kanunu.json"

print("PDF okunuyor ve sadece Kira Sözleşmesi bölümü aranıyor...")
belge = fitz.open(pdf_yolu)
tum_metin = ""

# Adım 1: Tüm PDF'i devasa bir metin bloğu haline getir
for sayfa in belge:
    tum_metin += sayfa.get_text("text") + "\n"

# Adım 2: Kira Sözleşmesi başlangıç (299) ve bitiş (379) noktalarını bul
baslangic = re.search(r"(?i)MADDE\s+299\s*[-–—]", tum_metin)
bitis = re.search(r"(?i)MADDE\s+379\s*[-–—]", tum_metin)

if baslangic and bitis:
    # Metni tam 299'un başladığı yerden, 379'un başladığı yere kadar kesiyoruz
    kira_metni = tum_metin[baslangic.start() : bitis.start()]
    print("✓ Kira Sözleşmesi bölümü (Madde 299 - 378 arası) başarıyla kırpıldı.")
else:
    print("HATA: Madde 299 veya Madde 379 PDF içinde bulunamadı!")
    exit()

# Adım 3: Kırpılmış metin içinde maddeleri tek tek ayır
sablon = r"(?i)MADDE\s+(\d+[A-Za-z]*)\s*[-–—]\s*"
parcalar = re.split(sablon, kira_metni)

maddeler = []

for i in range(1, len(parcalar)-1, 2):
    madde_no = parcalar[i]
    icerik = parcalar[i+1].strip()
    
    # PDF'ten kaynaklı gereksiz satır atlamalarını (enter) temizleyip tek satır yapıyoruz
    # Bu Qdrant'ın metni daha iyi anlamasını (embed etmesini) sağlar
    icerik = re.sub(r'\s+', ' ', icerik)
    
    ilk_cumle = icerik[:60]
    
    maddeler.append({
        "madde_no": madde_no,
        "baslik": f"TBK Madde {madde_no}",
        "icerik": icerik
    })

# Adım 4: Sadece bu maddeleri JSON olarak kaydet
with open(json_yolu, "w", encoding="utf-8") as f:
    json.dump(maddeler, f, ensure_ascii=False, indent=4)

print(f"İşlem tamam! Sadece Kira Hukukuna ait toplam {len(maddeler)} madde '{json_yolu}' dosyasına kaydedildi.")

In [ ]:
import json

# Colab upload kodlarını sildik, doğrudan senin dosyanın adını veriyoruz
dosya_adi = "tbk borclar kanunu.json" 

# Dosyayı Türkçe karakterleri bozmadan (utf-8) okuyoruz
with open(dosya_adi, 'r', encoding='utf-8') as f:
    maddeler = json.load(f)

print(f'✓ {len(maddeler)} madde yüklendi')
print(f'\nÖrnek madde:')
print(f"  Madde {maddeler[0]['madde_no']} — {maddeler[0]['baslik']}")
print(f"  İçerik: {maddeler[0]['icerik'][:100]}...")

## Hücre 4 — Embedding Modelini Yükle
`multilingual-e5-base` Türkçe dahil 100+ dili destekler.
İlk çalıştırmada model indirilir (~1GB) — birkaç dakika sürer.

In [ ]:
%pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

print('Model yükleniyor (ilk seferde ~1-2 dk)...')
# device="mps" ekleyerek Mac'in gücünü devreye sokuyoruz
model = SentenceTransformer('intfloat/multilingual-e5-base', device="mps")

print(f'✓ Model yüklendi')
print(f'  Embedding boyutu: {model.get_sentence_embedding_dimension()}')

# Hızlı test
test = model.encode(['passage: Kiracının hakları nelerdir?'])
print(f'  Test vektör boyutu: {test.shape}')

## Hücre 5 — Qdrant Koleksiyonu Oluştur

In [ ]:
%pip install qdrant-client

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# 1. Bağlantı ayarlarını locale göre güncelledik (API Key sildik)
QDRANT_URL = "http://localhost:6333"
KOLEKSIYON_ADI = "kira_hukuku"

client = QdrantClient(url=QDRANT_URL)

# Mevcut koleksiyonları kontrol et
mevcut = [c.name for c in client.get_collections().collections]
print(f'Mevcut koleksiyonlar: {mevcut}')

if KOLEKSIYON_ADI in mevcut:
    print(f'⚠ "{KOLEKSIYON_ADI}" zaten var — silip yeniden oluşturuluyor')
    client.delete_collection(KOLEKSIYON_ADI)

# Koleksiyonu E5 modeline uygun boyutta (768) oluştur
client.create_collection(
    collection_name=KOLEKSIYON_ADI,
    vectors_config=VectorParams(
        size=768,             # multilingual-e5-base boyutu
        distance=Distance.COSINE
    )
)

print(f'✓ Koleksiyon oluşturuldu: "{KOLEKSIYON_ADI}"')

## Hücre 6 — Embedding Üret ve Qdrant'a Yükle
78 madde için ~30-60 saniye sürer.

In [ ]:
import json
import time
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, Distance, VectorParams
from sentence_transformers import SentenceTransformer

# 1. Qdrant Bağlantısı (Localhost)
QDRANT_URL = "http://localhost:6333"
KOLEKSIYON_ADI = "kira_hukuku"

client = QdrantClient(url=QDRANT_URL)

# Mevcut koleksiyonu kontrol et ve gerekirse sıfırla
mevcut = [c.name for c in client.get_collections().collections]
if KOLEKSIYON_ADI in mevcut:
    print(f'⚠ "{KOLEKSIYON_ADI}" zaten var — silip yeniden oluşturuluyor')
    client.delete_collection(KOLEKSIYON_ADI)

# E5 Modeline uygun koleksiyon oluştur
client.create_collection(
    collection_name=KOLEKSIYON_ADI,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

# 2. Embedding Modelini Yükle (Mac Hızlandırması ile)
print('Model yükleniyor (ilk seferde ~1-2 dk)...')
model = SentenceTransformer('intfloat/multilingual-e5-base', device="mps")
print('✓ Model yüklendi')

# 3. JSON Verisini Oku
dosya_adi = "tbk borclar kanunu.json"
with open(dosya_adi, "r", encoding="utf-8") as f:
    maddeler = json.load(f)

print(f"✓ JSON okundu. Toplam {len(maddeler)} madde işlenecek.")

# 4. Yükleme Fonksiyonu
def maddeleri_yukle(maddeler, model, client, koleksiyon):
    # E5 modeli kuralı: Başına 'passage: ' eklemek zorunludur
    metinler = [f"passage: {m['baslik']} - {m['icerik']}" for m in maddeler]

    print(f'\n{len(metinler)} madde için embedding üretiliyor...')
    baslangic = time.time()
    
    # Metinleri vektörleştir
    vektorler = model.encode(
        metinler,
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    sure = time.time() - baslangic
    print(f'✓ Embedding tamamlandı ({sure:.1f} sn)')

    # Qdrant'a gönderilecek paketleri (PointStruct) hazırla
    points = [
        PointStruct(
            id=idx,
            vector=vektorler[idx].tolist(),
            payload={
                'madde_no': m['madde_no'],
                'baslik'  : m['baslik'],
                'icerik'  : m['icerik']
                # JSON dosyamızda url, kaynak vs. olmadığı için sadece bunları koyuyoruz
            }
        )
        for idx, m in enumerate(maddeler)
    ]

    # Batch (Toplu) Yükleme
    print('Qdrant\'a yükleniyor...')
    client.upsert(collection_name=koleksiyon, points=points)
    print(f'✓ {len(points)} madde Qdrant\'a başarıyla yüklendi!')

# 5. Fonksiyonu Çalıştır
maddeleri_yukle(maddeler, model, client, KOLEKSIYON_ADI)

In [ ]:
%pip install groq

In [ ]:
from groq import Groq
from qdrant_client.models import PayloadSchemaType

# Colab userdata yerine API anahtarını buraya tırnak içinde doğrudan yazmalısın
GROQ_API_KEY = "ap_keyini_buraya_yapistir"  # Secrets'tan güvenli okuma yapmalısın

groq_client = Groq(api_key=GROQ_API_KEY)

# qdrant değişken adını client yaptık ve INTEGER'ı KEYWORD yaptık
client.create_payload_index(
    collection_name="kira_hukuku",
    field_name="madde_no",
    field_schema=PayloadSchemaType.KEYWORD
)

print("✓ Groq hazır, index oluşturuldu")

In [ ]:
import re

def temizle(metin: str) -> str:
    metin = re.sub(r'\d{1,3}$', '', metin.strip())
    metin = re.sub(r'\d{1,2}\s+\d{1,2}/\d{1,2}/\d{4}', '', metin)
    metin = re.sub(r'\s+', ' ', metin).strip()
    return metin

for m in maddeler:
    m['baslik']    = temizle(m['baslik'])
    m['icerik']    = temizle(m['icerik'])
    m['tam_metin'] = f"Türk Borçlar Kanunu Madde {m['madde_no']} — {m['baslik']}\n\n{m['icerik']}"

print("✓ Temizlendi")
print(f"Madde 344 başlık: {next(m for m in maddeler if m['madde_no']==344)['baslik']}")

In [ ]:
from qdrant_client.models import PointStruct

metinler  = [f"passage: {m['tam_metin']}" for m in maddeler]
vektorler = model.encode(metinler, batch_size=16,
                         show_progress_bar=True, normalize_embeddings=True)

points = [
    PointStruct(id=idx, vector=vektorler[idx].tolist(), payload=m)
    for idx, m in enumerate(maddeler)
]
qdrant.upsert(collection_name=KOLEKSIYON_ADI, points=points)
print(f"✓ {len(points)} madde güncellendi")

In [ ]:
def sorgu_genislet(soru: str) -> str:
    yanit = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""Aşağıdaki soruyu Türk Borçlar Kanunu terminolojisiyle yeniden yaz.
Sadece yeniden yazılmış soruyu döndür, açıklama ekleme.
Soru: {soru}"""
        }],
        temperature=0, max_tokens=100,
    )
    return yanit.choices[0].message.content.strip()

def sor(soru: str):
    hukuki_soru = sorgu_genislet(soru)
    print(f"🔄 Hukuki sorgu: {hukuki_soru}")

    v1 = model.encode(f"query: {soru}", normalize_embeddings=True).tolist()
    v2 = model.encode(f"query: {hukuki_soru}", normalize_embeddings=True).tolist()

    # BURASI DÜZELTİLDİ: qdrant yerine client yazıldı
    s1 = client.query_points(collection_name=KOLEKSIYON_ADI, query=v1, limit=10, with_payload=True).points
    s2 = client.query_points(collection_name=KOLEKSIYON_ADI, query=v2, limit=10, with_payload=True).points

    goruldu, birlesik = set(), []
    for s in sorted(s1 + s2, key=lambda x: x.score, reverse=True):
        if s.payload['madde_no'] not in goruldu:
            goruldu.add(s.payload['madde_no'])
            birlesik.append(s)

    sonuclar = birlesik[:8]
    baglam   = "\n\n".join([
        f"Madde {s.payload['madde_no']} — {s.payload['baslik']}:\n{s.payload['icerik']}"
        for s in sonuclar
    ])

    yanit = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """Sen Türk kira hukuku asistanısın.
Verilen TBK maddelerinden soruyla doğrudan ilgili olanları kullan.
Her zaman madde numarasını belirt. Uydurma yapma."""
            },
            {
                "role": "user",
                "content": f"Maddeler:\n{baglam}\n\n---\nSoru: {soru}"
            }
        ],
        temperature=0.1, max_tokens=1024,
    )

    print(f"\n❓ Soru: {soru}")
    print("─" * 60)
    print(yanit.choices[0].message.content)
    print("\n📖 Başvurulan maddeler:", [s.payload['madde_no'] for s in sonuclar])

In [ ]:
sor("Ev sahibi kirayı ne kadar artırabilir?")
sor("Depozito ne zaman iade edilir?")
sor("Ev sahibi kiracıyı ne zaman çıkarabilir?")